In [6]:
"""
DESAFIO AC-1 - MOTOR DE DECISIONING SD-WAN ZERO-TRUST

Arquivo: desafio_ac1_master_sdwan.py

RELATÓRIO TÉCNICO

A solução utiliza um Algoritmo Genético para selecionar uma rota entre
o nó de origem 0 e o nó de destino 11 em uma topologia full-mesh com
12 roteadores.

A função de fitness utilizada é:

    Fitness(X) =
        w1 * LatenciaTotal(X)
        + w2 * PerdaPacotesTotal(X)
        + P_Seguranca

Foram adotados:
    w1 = 1.0
    w2 = 10.0
    P_Seguranca = 5000

A semente estocástica utilizada é np.random.seed(2026).

Com essa configuração, a melhor rota encontrada é:

    0 -> 1 -> 4 -> 11

Resultados da rota:
    Latência total: 47.00 ms
    Perda total de pacotes: 1.80 %
    Penalidade de segurança: 0.00
    Fitness final: 65.00

Os nós 2, 5 e 8 possuem reputação inferior a 50 e, portanto, são
considerados não confiáveis.

A rota insegura 0 -> 2 -> 5 -> 11 apresenta latência e perda menores,
mas passa pelos nós 2 e 5. Por esse motivo recebe a penalização de
segurança de 5000, deixando de ser atrativa para o algoritmo.

Assim, a solução selecionada evita os nós penalizados e prioriza uma
rota segura, mesmo abrindo mão da menor latência bruta possível.
"""

import time
import numpy as np


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

np.random.seed(2026)

NUM_NOS = 12
ORIGEM = 0
DESTINO = 11

TAMANHO_POPULACAO = 120
NUM_GERACOES = 220
TAXA_MUTACAO = 0.30
TAXA_IMIGRACAO = 0.10

W_LATENCIA = 1.0
W_PERDA = 10.0

LIMITE_REPUTACAO = 50.0
PENALIDADE_SEGURANCA = 5000.0

NOS_INTERMEDIARIOS = np.arange(1, 11)


# ============================================================
# 2. TOPOLOGIA E ATRIBUTOS
# ============================================================

# Topologia full-mesh:
# 1 = existe enlace entre os nós
# 0 = não existe enlace
matriz_adjacencia = np.ones((NUM_NOS, NUM_NOS), dtype=int)
np.fill_diagonal(matriz_adjacencia, 0)


def criar_matriz_simetrica(valor_minimo, valor_maximo):
    matriz = np.random.uniform(
        valor_minimo,
        valor_maximo,
        (NUM_NOS, NUM_NOS)
    )

    matriz = (matriz + matriz.T) / 2
    np.fill_diagonal(matriz, 0)

    return matriz


matriz_latencia = criar_matriz_simetrica(80, 130)
matriz_perda = criar_matriz_simetrica(2, 6)


def configurar_enlace(origem, destino, latencia, perda):
    matriz_latencia[origem, destino] = latencia
    matriz_latencia[destino, origem] = latencia

    matriz_perda[origem, destino] = perda
    matriz_perda[destino, origem] = perda


# Caminho seguro
configurar_enlace(0, 1, 18, 0.7)
configurar_enlace(1, 4, 15, 0.6)
configurar_enlace(4, 11, 14, 0.5)

# Caminho muito rápido, porém inseguro
configurar_enlace(0, 2, 6, 0.1)
configurar_enlace(2, 5, 6, 0.1)
configurar_enlace(5, 11, 6, 0.1)

# Enlace direto propositalmente menos vantajoso
configurar_enlace(0, 11, 150, 8.0)


# Reputação dos 12 roteadores
reputacao = np.array([
    95,  # Nó 0
    88,  # Nó 1
    35,  # Nó 2 - não confiável
    92,  # Nó 3
    84,  # Nó 4
    42,  # Nó 5 - não confiável
    90,  # Nó 6
    86,  # Nó 7
    28,  # Nó 8 - não confiável
    81,  # Nó 9
    93,  # Nó 10
    97   # Nó 11
], dtype=float)


# ============================================================
# 3. REPRESENTAÇÃO DO INDIVÍDUO
# ============================================================

def criar_individuo():
    ordem = np.random.permutation(NOS_INTERMEDIARIOS)

    quantidade = int(
        np.random.randint(
            0,
            len(NOS_INTERMEDIARIOS) + 1
        )
    )

    return {
        "ordem": ordem,
        "quantidade": quantidade
    }


def copiar_individuo(individuo):
    return {
        "ordem": individuo["ordem"].copy(),
        "quantidade": int(individuo["quantidade"])
    }


def obter_rota(individuo):
    quantidade = individuo["quantidade"]

    intermediarios = individuo["ordem"][:quantidade]

    return np.concatenate((
        [ORIGEM],
        intermediarios,
        [DESTINO]
    )).astype(int)


# ============================================================
# 4. CÁLCULO DA LATÊNCIA E PERDA
# ============================================================

def calcular_latencia_total(rota):
    total = 0.0

    for i in range(len(rota) - 1):
        origem = rota[i]
        destino = rota[i + 1]

        total += matriz_latencia[origem, destino]

    return total


def calcular_perda_total(rota):
    total = 0.0

    for i in range(len(rota) - 1):
        origem = rota[i]
        destino = rota[i + 1]

        total += matriz_perda[origem, destino]

    return total


# ============================================================
# 5. PENALIZAÇÃO ZERO-TRUST
# ============================================================

def calcular_penalidade_seguranca(rota):
    for no in rota:
        if reputacao[no] < LIMITE_REPUTACAO:
            return PENALIDADE_SEGURANCA

    return 0.0


# ============================================================
# 6. FUNÇÃO FITNESS
# ============================================================

def calcular_fitness(individuo):
    rota = obter_rota(individuo)

    latencia = calcular_latencia_total(rota)
    perda = calcular_perda_total(rota)
    penalidade = calcular_penalidade_seguranca(rota)

    return (
        W_LATENCIA * latencia
        + W_PERDA * perda
        + penalidade
    )


# ============================================================
# 7. CROSSOVER OX
# ============================================================

def crossover_ox(pai1, pai2):
    tamanho = len(pai1)

    filho = np.full(
        tamanho,
        -1,
        dtype=int
    )

    ponto1, ponto2 = sorted(
        np.random.choice(
            tamanho,
            2,
            replace=False
        )
    )

    filho[ponto1:ponto2] = pai1[ponto1:ponto2]

    elementos_pai2 = np.concatenate((
        pai2[ponto2:],
        pai2[:ponto2]
    ))

    posicao = ponto2

    for elemento in elementos_pai2:
        if elemento not in filho:
            if posicao >= tamanho:
                posicao = 0

            filho[posicao] = elemento
            posicao += 1

    return filho


def crossover(pai1, pai2):
    nova_ordem = crossover_ox(
        pai1["ordem"],
        pai2["ordem"]
    )

    if np.random.rand() < 0.5:
        nova_quantidade = pai1["quantidade"]
    else:
        nova_quantidade = pai2["quantidade"]

    return {
        "ordem": nova_ordem,
        "quantidade": int(nova_quantidade)
    }


# ============================================================
# 8. MUTAÇÃO
# ============================================================

def mutacao(individuo):
    filho = copiar_individuo(individuo)

    if np.random.rand() < TAXA_MUTACAO:
        idx1, idx2 = np.random.choice(
            len(filho["ordem"]),
            2,
            replace=False
        )

        filho["ordem"][idx1], filho["ordem"][idx2] = (
            filho["ordem"][idx2],
            filho["ordem"][idx1]
        )

    if np.random.rand() < TAXA_MUTACAO:
        filho["quantidade"] = int(
            np.random.randint(
                0,
                len(NOS_INTERMEDIARIOS) + 1
            )
        )

    return filho


# ============================================================
# 9. SELEÇÃO POR TORNEIO
# ============================================================

def selecao_torneio(populacao, fitness, tamanho_torneio=3):
    participantes = np.random.randint(
        0,
        len(populacao),
        size=tamanho_torneio
    )

    melhor = min(
        participantes,
        key=lambda indice: fitness[indice]
    )

    return populacao[melhor]


# ============================================================
# 10. POPULAÇÃO INICIAL
# ============================================================

populacao = [
    criar_individuo()
    for _ in range(TAMANHO_POPULACAO)
]

melhor_individuo_global = None
melhor_fitness_global = np.inf

historico_melhor = []


# ============================================================
# 11. EXECUÇÃO DO ALGORITMO GENÉTICO
# ============================================================

tempo_inicio = time.time()

for geracao in range(NUM_GERACOES):

    fitness_populacao = [
        calcular_fitness(individuo)
        for individuo in populacao
    ]

    indice_melhor = int(
        np.argmin(fitness_populacao)
    )

    melhor_fitness_geracao = (
        fitness_populacao[indice_melhor]
    )

    if melhor_fitness_geracao < melhor_fitness_global:
        melhor_fitness_global = melhor_fitness_geracao

        melhor_individuo_global = copiar_individuo(
            populacao[indice_melhor]
        )

    historico_melhor.append(
        melhor_fitness_global
    )

    # Elitismo: preserva a melhor solução já encontrada
    nova_populacao = [
        copiar_individuo(melhor_individuo_global)
    ]

    while len(nova_populacao) < TAMANHO_POPULACAO:

        # Pequena inserção de novos indivíduos aleatórios
        # para aumentar a diversidade da população
        if np.random.rand() < TAXA_IMIGRACAO:
            nova_populacao.append(
                criar_individuo()
            )
            continue

        pai1 = selecao_torneio(
            populacao,
            fitness_populacao
        )

        pai2 = selecao_torneio(
            populacao,
            fitness_populacao
        )

        filho = crossover(
            pai1,
            pai2
        )

        filho = mutacao(
            filho
        )

        nova_populacao.append(
            filho
        )

    populacao = nova_populacao

tempo_fim = time.time()


# ============================================================
# 12. RESULTADOS
# ============================================================

melhor_rota = obter_rota(
    melhor_individuo_global
)

latencia_final = calcular_latencia_total(
    melhor_rota
)

perda_final = calcular_perda_total(
    melhor_rota
)

penalidade_final = calcular_penalidade_seguranca(
    melhor_rota
)

fitness_final = calcular_fitness(
    melhor_individuo_global
)

tempo_execucao = (
    tempo_fim - tempo_inicio
) * 1000


nos_nao_confiaveis = [
    no
    for no in range(NUM_NOS)
    if reputacao[no] < LIMITE_REPUTACAO
]


print("\n" + "=" * 65)
print("DESAFIO AC-1 - SD-WAN ZERO-TRUST")
print("=" * 65)

print(
    "Melhor rota:",
    " -> ".join(
        map(str, melhor_rota)
    )
)

print(
    f"Latência total: {latencia_final:.2f} ms"
)

print(
    f"Perda total: {perda_final:.2f} %"
)

print(
    f"Penalidade de segurança: {penalidade_final:.2f}"
)

print(
    f"Fitness final: {fitness_final:.2f}"
)

print(
    "Nós não confiáveis:",
    nos_nao_confiaveis
)

print(
    f"Tempo de execução: {tempo_execucao:.2f} ms"
)

print("=" * 65)


print("\nREPUTAÇÃO DOS NÓS DA ROTA")

for no in melhor_rota:
    situacao = (
        "CONFIÁVEL"
        if reputacao[no] >= LIMITE_REPUTACAO
        else "NÃO CONFIÁVEL"
    )

    print(
        f"Nó {no}: "
        f"reputação={reputacao[no]:.0f} "
        f"({situacao})"
    )


# ============================================================
# 13. COMPARAÇÃO COM UMA ROTA INSEGURA
# ============================================================

rota_insegura = np.array([
    0,
    2,
    5,
    11
])

latencia_insegura = calcular_latencia_total(
    rota_insegura
)

perda_insegura = calcular_perda_total(
    rota_insegura
)

penalidade_insegura = calcular_penalidade_seguranca(
    rota_insegura
)

fitness_insegura = (
    W_LATENCIA * latencia_insegura
    + W_PERDA * perda_insegura
    + penalidade_insegura
)


print("\nCOMPARAÇÃO COM ROTA INSEGURA")

print(
    "Rota:",
    " -> ".join(
        map(str, rota_insegura)
    )
)

print(
    f"Latência: {latencia_insegura:.2f} ms"
)

print(
    f"Perda: {perda_insegura:.2f} %"
)

print(
    f"Penalidade: {penalidade_insegura:.2f}"
)

print(
    f"Fitness: {fitness_insegura:.2f}"
)



DESAFIO AC-1 - SD-WAN ZERO-TRUST
Melhor rota: 0 -> 1 -> 4 -> 11
Latência total: 47.00 ms
Perda total: 1.80 %
Penalidade de segurança: 0.00
Fitness final: 65.00
Nós não confiáveis: [2, 5, 8]
Tempo de execução: 5412.05 ms

REPUTAÇÃO DOS NÓS DA ROTA
Nó 0: reputação=95 (CONFIÁVEL)
Nó 1: reputação=88 (CONFIÁVEL)
Nó 4: reputação=84 (CONFIÁVEL)
Nó 11: reputação=97 (CONFIÁVEL)

COMPARAÇÃO COM ROTA INSEGURA
Rota: 0 -> 2 -> 5 -> 11
Latência: 18.00 ms
Perda: 0.30 %
Penalidade: 5000.00
Fitness: 5021.00
